# 📍 Indoor Position Prediction Using Machine Learning from BLE Signal Fingerprints

### Comprehensive 12-Stage Machine Learning Pipeline
**Dataset**: BLE RSSI Dataset for Indoor Localization and Navigation (UCI ML Repository / Waldo Library)
**Authors / Topic**: Multiclass classification, feature engineering, noisy-data handling, physical localization error analysis.

---
### Workflow Overview
1. **Dataset Understanding**
2. **Data Cleaning & Missing/Sentinel Value Treatment**
3. **Exploratory Data Analysis (EDA)**
4. **Feature Engineering**
5. **Feature Selection & Dimensionality Reduction**
6. **Baseline ML Model**
7. **Proposed / Improved ML Model**
8. **Hyperparameter Optimization & Ensemble Systems**
9. **Model Evaluation (Classification & Physical Distance Error)**
10. **Explainability & Spatial Error Diagnosis (SHAP & Permutation Importance)**
11. **Noise & Packet Drop Robustness Experiments**
12. **Final ML Prediction System & Out-of-Sample Inference**

In [ ]:
# Core Imports
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn, XGBoost, SHAP
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
import xgboost as xgb
import shap

# Configure Visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("All libraries successfully loaded.")

## 1. Dataset Understanding
We inspect the labeled dataset collected in Waldo Library at Western Michigan University.
It consists of RSSI measurements from 13 BLE iBeacons (`b3001` through `b3013`) recorded on an iPhone 6S.

In [ ]:
# Dynamic base path detection
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, '..')) if os.path.basename(CURRENT_DIR) == 'notebooks' else os.path.abspath(CURRENT_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'iBeacon_RSSI_Labeled.csv')
df_labeled = pd.read_csv(DATA_PATH)
print(f"Loaded dataset from: {DATA_PATH}")
print(f"Dataset Shape: {df_labeled.shape}")
print(f"Unique Indoor Grid Locations: {df_labeled['location'].nunique()}")
display(df_labeled.head())

## 2. Data Cleaning & Out-of-Range Sentinel Treatment
In BLE fingerprinting, an RSSI value of `-200 dBm` denotes that a beacon is out of transmission range.
We map symbolic location strings (`K04`, `D13`) to 2D physical Cartesian coordinates `(x, y)` to enable exact distance error calculation.

In [ ]:
import re
def parse_loc(loc_str):
    match = re.match(r'^([A-Za-z]+)(\d+)$', str(loc_str).strip())
    if not match:
        return None, None
    col_str, row_str = match.groups()
    return ord(col_str.upper()) - ord('A'), int(row_str)

coords = [parse_loc(loc) for loc in df_labeled['location']]
df_labeled['coord_x'] = [c[0] for c in coords]
df_labeled['coord_y'] = [c[1] for c in coords]

beacon_cols = [f'b30{i:02d}' for i in range(1, 14)]
print("Check Missing Values (NaN):", df_labeled[beacon_cols].isna().sum().sum())
display(df_labeled[['location', 'coord_x', 'coord_y'] + beacon_cols[:4]].head())

## 3. Exploratory Data Analysis (EDA)
Visualizing beacon coverage, signal strength distributions, and physical spatial density.

In [ ]:
# Beacon Detection Frequency (% of samples in range)
det_rates = [(df_labeled[col] > -200).mean() * 100 for col in beacon_cols]

plt.figure(figsize=(10, 4))
sns.barplot(x=beacon_cols, y=det_rates, color='#1f77b4')
plt.title('Beacon In-Range Detection Rates (%)', fontweight='bold')
plt.xlabel('iBeacon ID')
plt.ylabel('Detection Rate (%)')
plt.xticks(rotation=45)
plt.show()

## 4. Feature Engineering
Creating domain-specific radio propagation features:
- Number of active visible beacons
- Maximum and minimum detected RSSI
- Signal mean, spread, and standard deviation
- Top-2 beacon RSSI delta (spatial proximity indicator)
- Linear power sum $\sum 10^{(\text{RSSI}/10)}$
- Normalized non-negative signal scales

In [ ]:
sys.path.append('..')
from src.feature_engineering import engineer_features, get_feature_names
from src.preprocessing import PreprocessingPipeline, prepare_train_test_split

train_df, test_df = prepare_train_test_split(df_labeled, test_size=0.2, random_state=42)
pipeline = PreprocessingPipeline(scaler_type='standard', include_engineered=True)
X_train, y_train = pipeline.fit_transform(train_df)
X_test, y_test = pipeline.transform(test_df)

print(f'Train samples: {len(X_train)} | Test samples: {len(X_test)}')
print(f'Engineered Feature Count: {X_train.shape[1]}')

## 5. Feature Selection & Influence Analysis
Evaluating feature influence using Mutual Information and Random Forest Gini Importance.

In [ ]:
from src.feature_selection import compute_feature_importances

importance_df = compute_feature_importances(X_train, y_train, pipeline.feature_names)
display(importance_df.head(10))

## 6, 7 & 8. Model Training & Ensembles
Comparing Baselines, Optimized Standalone Models, and Multi-Model Ensembles.

In [ ]:
from src.models import get_baseline_models, get_candidate_models, build_ensemble_models

# Baselines on raw RSSI
raw_pipeline = PreprocessingPipeline(scaler_type='standard', include_engineered=False)
X_train_raw, y_train_raw = raw_pipeline.fit_transform(train_df)
X_test_raw, y_test_raw = raw_pipeline.transform(test_df)

models_dict = {}

# Train Baselines
for name, clf in get_baseline_models().items():
    clf.fit(X_train_raw, y_train_raw)
    models_dict[name] = (clf, X_test_raw)

# Train Candidates & Ensembles
candidates = get_candidate_models(random_state=42)
for name, clf in candidates.items():
    print(f'Training {name}...')
    clf.fit(X_train, y_train)
    models_dict[name] = (clf, X_test)

# Ensembles
ensembles = build_ensemble_models(candidates, random_state=42)
for name, clf in ensembles.items():
    print(f'Training {name}...')
    clf.fit(X_train, y_train)
    models_dict[name] = (clf, X_test)

print('All models successfully trained!')

## 9. Comparative Model Evaluation
Evaluating classification accuracy and physical Euclidean distance localization error across all algorithms.

In [ ]:
from src.evaluation import evaluate_predictions

records = []
for name, (clf, X_eval) in models_dict.items():
    preds = clf.predict(X_eval)
    proba = clf.predict_proba(X_eval) if hasattr(clf, 'predict_proba') else None
    metrics = evaluate_predictions(y_test, preds, pipeline.class_to_coords, proba)
    metrics['model'] = name
    records.append(metrics)

results_df = pd.DataFrame(records)[['model', 'accuracy', 'top3_accuracy', 'mean_distance_error', 'median_distance_error', 'within_1_grid_acc', 'f1_weighted']]
results_df = results_df.sort_values(by='mean_distance_error', ascending=True).reset_index(drop=True)
display(results_df)

## 10. Explainability & Error Diagnosis
Interpreting model predictions using Permutation Importance and mapping physical grid errors.

In [ ]:
from src.explainability import run_permutation_importance

best_model_name = results_df.iloc[0]['model']
best_clf, X_eval = models_dict[best_model_name]
perm_df = run_permutation_importance(best_clf, X_eval, y_test, pipeline.feature_names)
display(perm_df.head(10))

## 11. Noise & Packet Drop Robustness Experiment
Simulating real-world RF channel impairments (multipath jitter and packet drops).

In [ ]:
from src.noise_experiment import evaluate_noise_robustness

eval_dict = {k: v[0] for k, v in models_dict.items() if 'Dummy' not in k and 'Baseline' not in k}
noise_df = evaluate_noise_robustness(eval_dict, pipeline, test_df, noise_levels=[0.0, 3.0, 6.0, 10.0, 15.0])
display(noise_df.head(10))

## 12. Final Prediction System: Out-of-Sample Unlabeled Demonstration

In [ ]:
UNLAB_PATH = os.path.join(PROJECT_ROOT, 'data', 'iBeacon_RSSI_Unlabeled.csv')
df_unlabeled = pd.read_csv(UNLAB_PATH)
X_unlab, _ = pipeline.transform(df_unlabeled.head(5))
unlab_preds = best_clf.predict(X_unlab)
unlab_locations = pipeline.inverse_transform_labels(unlab_preds)

for i, loc in enumerate(unlab_locations):
    c = pipeline.class_to_coords[unlab_preds[i]]
    print(f"Unlabeled Sample #{i+1} [Timestamp: {df_unlabeled['date'].iloc[i]}]: Predicted Location = {loc} (Grid: x={c[0]}, y={c[1]})")